# Scaling ML Workflows: PySpark ETL on WikiText & Cloud TPU miniGPT Training

### An end-to-end distributed Machine Learning pipeline on Google Kubernetes Engine (GKE)

This notebook demonstrates scaling an ML workflow from local debugging to full multi-node distribution without leaving the Jupyter interface. We will:

1. Use **Apache Spark** to distribute the preprocessing of a WikiText dataset into chunked Token IDs.
2. Prototype a **miniGPT Language Model in JAX/Flax** on our local Jupyter TPU slices.
3. Scale out to a **Multi-Host TPU v5p Slice** for data-parallel JAX training using Kubeflow Trainer.
4. Deploy the trained model to an **NVIDIA GPU** backend for high-throughput text generation.

## 0. Setup & environment check

We verify that our Python SDKs are loaded, check that our Workspace is connected to the GKE Cloud TPU hardware, and insert our jobs folder into the Python path.

In [ ]:
import os
import sys
import subprocess

# Ensure the jobs package is on the import path
DEMO_DIR = os.getcwd()
if not os.path.isdir(os.path.join(DEMO_DIR, "jobs")):
    DEMO_DIR = "/home/jovyan/demo"
if DEMO_DIR not in sys.path:
    sys.path.insert(0, DEMO_DIR)

import jax
import kubeflow.trainer
import kubeflow.spark

print("JAX version:", jax.__version__)
print("Local TPU cores detected on this notebook VM:", jax.local_device_count())
print("Local TPU devices:", jax.local_devices())

# Verify the notebook's permissions
for res in ["trainjobs.trainer.kubeflow.org", "sparkapplications.sparkoperator.k8s.io"]:
    out = subprocess.run(["kubectl", "auth", "can-i", "create", res],
                         capture_output=True, text=True)
    print(f"Can-i create {res.split('.')[0]:15s}:", out.stdout.strip() or out.stderr.strip())

### Configure GCS storage

We use a dedicated **GCS bucket** as our shared storage bus. Spark executors will write processed training shards to this bucket, which the JAX TPU trainer will read.

In [ ]:
bucket_name = os.environ.get("DEMO_BUCKET", "sizhang-gke-dev-ml-demo-data")
print(f"Using shared GCS bucket: gs://{bucket_name}")

## Stage 1 — Distributed WikiText Processing with Apache Spark

The WikiText dataset is processed using Spark. The job downloads the raw data, tokenizes it using a HuggingFace GPT2 tokenizer, and generates sequence chunks. The Spark executors write the processed `.npz` shards directly to our shared GCS bucket.

In [ ]:
import subprocess
from jobs import pipeline

# Clean up any stale Spark ETL jobs from previous runs
subprocess.run(["kubectl", "delete", "sparkconnect", "scaling-data-etl", "--ignore-not-found"])

# Submit the distributed Spark ETL job and wait for completion
pipeline.run_data_processing(num_executors=4, num_shards=4, num_records=20000, block_size=128)

In [ ]:
# Print Spark driver logs to verify execution details
pipeline.print_spark_logs()

## Stage 2 — Local TPU Interactive Debugging

Before executing a massive distributed training run across the cluster, we should verify our model logic. We execute our miniGPT training function locally on the Jupyter Workspace's dedicated TPU cores.

We can adjust the hyperparameters here to train a very small model (e.g., 2 layers) on a subset of the data just to ensure it converges.

In [ ]:
# Set environmental configurations for local debug
os.environ["BUCKET_NAME"] = bucket_name
os.environ["EPOCHS"] = "2"
os.environ["GLOBAL_BATCH_SIZE"] = "8"
os.environ["BLOCK_SIZE"] = "128"
os.environ["VOCAB_SIZE"] = "50257"
os.environ["N_LAYER"] = "2"
os.environ["N_HEAD"] = "2"
os.environ["N_EMBD"] = "64"

from jobs.train import train_scaling_model

# Run locally
train_scaling_model(is_local_debug=True)

## Stage 3 — Full-scale Distributed TPU Training

Now that our JAX training loop is verified, we are ready to scale out. We will train a larger miniGPT model by submitting a `TrainJob` that spawns across an entire **2-host TPU v5p slice**.

The `pipeline.run_training` helper creates the `TrainJob` Custom Resource using the Kubeflow Trainer Python SDK.

In [ ]:
import subprocess

# Release TPU nodes held by the reservation placeholder
subprocess.run(["kubectl", "delete", "job", "tpu-job-ccc", "-n", "default", "--ignore-not-found"])

In [ ]:
# Print multi-host TrainJob logs and load final training metrics from GCS
train_job_name = pipeline.run_training(
    num_hosts=2,
    epochs=10, 
    global_batch_size=64, 
    block_size=128, 
    vocab_size=50257, 
    n_layer=4, 
    n_head=4, 
    n_embd=128, 
    wait=True
)
pipeline.print_logs(train_job_name)

## Stage 4 — GPU-Accelerated Inference & Model Serving

With the final weights (`params.npz`) and metrics in GCS, we can deploy the model. While TPUs are excellent for data-parallel training, NVIDIA GPUs (like the L4) are often preferred for serving. 

We submit a Kubernetes Deployment that:
1. Provisions an L4 GPU node.
2. Runs our `serve.py` script.
3. Automatically downloads the Flax weights from GCS and converts them to PyTorch CUDA tensors for high-performance text generation.

In [ ]:
# Deploy inference service using the python script helper
pipeline.deploy_inference()

### Test predictions

We send an HTTP POST request containing a prompt to our inference service, and the PyTorch backend generates text.

In [ ]:
import urllib.request
import json
import numpy as np

prompt = "The history of the world is"
payload = json.dumps({"prompt": prompt, "max_new_tokens": 50}).encode("utf-8")

req = urllib.request.Request(
    "http://scaling-model-inference.default.svc.cluster.local:80/generate",
    data=payload,
    headers={"Content-Type": "application/json"}
)

try:
    with urllib.request.urlopen(req) as response:
        result = json.loads(response.read().decode("utf-8"))
        print("Prompt:")
        print(result["prompt"])
        print("\nGenerated Text:")
        print(result["text"])
except Exception as e:
    print(f"Error querying inference service: {e}")

## Conclusion & Re-applying TPU Reservation

We have demonstrated a complete, scaled-up ML pipeline run entirely from a single notebook interface. 

To finish up, we re-apply the TPU reservation placeholder so that our Cloud TPU quota remains locked for our workspace's exclusive usage.

In [ ]:
import subprocess

# Re-apply the reservation placeholder job
subprocess.run(["kubectl", "apply", "-f", "../tpu-job-ccc.yaml"])